# simple_spread: the fixed implementation on a cooperative task

After the probes (`probe_paradigms.ipynb`), the second test of `core/rl/`, on a
task that is not ours: Farama's MPE2 `simple_spread`
([docs](https://mpe2.farama.org/mpe/simple_spread/)) with continuous actions. All
nine combinations (PPO, APPO, SAC × CTCE, CTDE, DTDE) are trained on **the same
budget of environment steps** and scored on **the same evaluation episodes**.

**What it adds to the probes.** In the probes each agent's reward depends only on
its own action. Here the agents share a team reward and get in each other's way,
so it is the first test where the agents interact. There is no exact answer, only
"better than the baselines": a random policy and the zero action (no force).

**What it does not show.** A combination that stays at the baseline level after
this budget has not necessarily failed: `simple_spread` usually needs more steps
than a notebook run gives it. The probes say whether the machinery is right; this
says how far each combination gets on a fixed budget.

**Running it.** With `QUICK = True` (the default) every combination trains a few
thousand steps, enough to see that everything runs. For the real run set
`QUICK = False` (100,000 steps each, roughly an hour with 3 workers). Runs are
written to `runs/spread_quick/` or `runs/spread/` and finished runs are not
trained again. Needs `pip install mpe2` in the `marlem` environment. The
original code's results are in `RL/notebooks/original/simple_spread.ipynb` on the `clean` branch.

In [ ]:
import json
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path

# Repository root: found from wherever the notebook runs, and put on PYTHONPATH.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "rl").is_dir())
sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(ROOT), os.environ.get("PYTHONPATH", "")]))  # the training processes import core.rl too

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as GifImage, display
from PIL import Image, ImageDraw

from core.rl.envs.spread import SimpleSpread, episode_frames, random_policy
from core.rl import Run, load_policy, train

pd.set_option("display.precision", 2)
pd.set_option("display.width", 160)

ENV_CONFIG = {"n_agents": 3, "max_cycles": 25, "local_ratio": 0.5}
# What to train: every combination of these (e.g. ALGORITHMS = ["ppo"] for PPO only).
ALGORITHMS = ["ppo"]  # V2: PPO only
PARADIGMS = ["ctce", "ctde", "dtde"]
SEED = 0

# True: a few thousand steps per combination, to check that everything runs.
# False: the real budget.
QUICK = True
BUDGET = 5_000 if QUICK else 100_000  # environment steps, the same for all nine
EVAL_EVERY = 0 if QUICK else 10_000  # evaluate during training every this many steps
EVAL_EPISODES = 5 if QUICK else 20  # evaluation episodes, on seeds training never uses

# On top of core/rl/config.SETTINGS (the paper's): learning rate 3e-4 instead of
# 1e-5, PPO batches of 1,000 steps, SAC 16 gradient updates per 256 sampled steps.
TRAINING = {
    "ppo": {"lr": 3e-4, "train_batch_size_per_learner": 1000},
    "appo": {"lr": 3e-4},
    "sac": {"actor_lr": 3e-4, "critic_lr": 3e-4, "training_intensity": 16},
}

WORKERS = 3  # parallel training processes
RUNS_DIR = ROOT / "runs" / ("spread_quick" if QUICK else "spread")

## a. The environment

Three agents and three landmarks in a plane; the agents should cover the
landmarks, one each, without bumping into each other. An episode lasts 25 steps
and ends by time (truncation).

- **Observation** (18 numbers per agent): own velocity (2), own position (2), the
  landmarks relative to the agent (3 × 2), the other agents relative to the agent
  (2 × 2), and the others' communication (2 × 2, always 0). Every agent sees
  everything, so CTCE's joint observation repeats information.
- **Action** (`Box(0, 1, (5,))` per agent): `[no-op, left, right, down, up]`; the
  agent is pushed with `(right − left, up − down)`. Equal values cancel, so the
  zero action stands still.
- **Reward** (per agent): half the team's reward (minus the distance from each
  landmark to its nearest agent, the same for everyone) plus half the agent's own
  (minus 1 per collision). The team score is the sum over the agents.

In [ ]:
env = SimpleSpread(ENV_CONFIG)
print("agents:", env.agents_id)
print("observation:", env.observation_spaces["agent_00"])
print("action:", env.action_spaces["agent_00"])


def gif(columns, labels, fps=8):
    """Frames of several episodes side by side, as one small GIF."""
    out = []
    for t in range(min(len(c) for c in columns)):
        panel = Image.fromarray(np.concatenate([c[t] for c in columns], axis=1))
        draw = ImageDraw.Draw(panel)
        width = columns[0][0].shape[1]
        for i, label in enumerate(labels):
            draw.text((i * width + 6, 4), label, fill=(0, 0, 0))
        out.append(panel.quantize(colors=32))
    buffer = BytesIO()
    out[0].save(buffer, format="GIF", save_all=True, append_images=out[1:],
                duration=int(1000 / fps), loop=0, optimize=True)
    return GifImage(data=buffer.getvalue(), format="gif")


frames_random, return_random = episode_frames(ENV_CONFIG, random_policy(ENV_CONFIG, 1), SEED)
print(f"one episode with random actions, team return {return_random:.1f}")
gif([frames_random], ["random actions"])

## b. Train the selected combinations

Every combination of `ALGORITHMS` and `PARADIGMS` (set in the first code cell) is one `core.rl.train` run with the same budget of
environment steps. Every run evaluates, on the same held-out seeds, the policy
before training, the random policy and the zero action (at step 0), then the
trained policy every `EVAL_EVERY` steps and at the end. The trained policy acts
with its most likely action, as an agent would after training.

In [ ]:
def make_run(algorithm: str, paradigm: str) -> Run:
    return Run(
        algorithm,
        paradigm,
        SimpleSpread,
        ENV_CONFIG,
        budget=BUDGET,
        seed=SEED,
        eval_every=EVAL_EVERY,
        eval_episodes=EVAL_EPISODES,
        training=TRAINING[algorithm],
        out_dir=RUNS_DIR / f"{algorithm}_{paradigm}",
    )


def status(run: Run) -> str:
    path = Path(run.out_dir) / "run.json"
    return json.loads(path.read_text())["status"] if path.exists() else "not run"


def train_all(runs: list[Run], workers: int = WORKERS) -> None:
    """Train the runs that have not finished, in parallel processes."""
    pending = [r for r in runs if status(r) != "completed"]
    slowest_first = {"sac": 0, "appo": 1, "ppo": 2}
    pending.sort(key=lambda r: (slowest_first[r.algorithm], r.paradigm != "dtde"))
    print(f"{len(runs)} runs, {len(pending)} to train")
    start = time.time()
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(train, run): run for run in pending}
        for future in as_completed(futures):
            run = futures[future]
            try:
                note = f"done in {future.result()['duration_s']:.0f} s"
            except Exception as error:
                note = f"FAILED: {error!r}"[:160]
            minutes = (time.time() - start) / 60
            print(f"{Path(run.out_dir).name:10s} {note} ({minutes:.0f} min)", flush=True)


RUNS = {(a, p): make_run(a, p) for a in ALGORITHMS for p in PARADIGMS}
train_all(list(RUNS.values()))

## c. Results

Team return per episode (sum over the three agents; higher is better, 0 is the
theoretical best), all on the same evaluation episodes: the trained policy at the
end, the same networks before training, and the two baselines.

In [ ]:
def evaluation(key) -> pd.DataFrame:
    return pd.read_csv(Path(RUNS[key].out_dir) / "evaluation.csv")


done = [key for key, run in RUNS.items() if status(run) == "completed"]
first = evaluation(done[0])
baselines = first[first.env_steps == 0].set_index("policy").team_return_mean
print(f"on {EVAL_EPISODES} evaluation episodes: random {baselines['random']:.1f}, "
      f"zero action {baselines['zero']:.1f}")

rows = {}
for key in RUNS:
    if key not in done:
        rows[key] = {"status": status(RUNS[key])}
        continue
    frame = evaluation(key)
    trained = frame[frame.policy == "trained"]
    rows[key] = {
        "untrained": trained.team_return_mean.iloc[0],
        "trained": trained.team_return_mean.iloc[-1],
        "trained std": trained.team_return_std.iloc[-1],
        "beats random": trained.team_return_mean.iloc[-1] > baselines["random"],
        "beats zero": trained.team_return_mean.iloc[-1] > baselines["zero"],
    }
table = pd.DataFrame(rows).T
table.index.names = ["algorithm", "paradigm"]
table

In [ ]:
fig, axes = plt.subplots(len(PARADIGMS), len(ALGORITHMS), squeeze=False,
                         figsize=(4.5 * len(ALGORITHMS), 2.7 * len(PARADIGMS)), sharey=True)
for row, paradigm in enumerate(PARADIGMS):
    for col, algorithm in enumerate(ALGORITHMS):
        ax, key = axes[row, col], (algorithm, paradigm)
        if key in done:
            metrics = pd.read_csv(Path(RUNS[key].out_dir) / "metrics.csv")
            ax.plot(metrics.env_steps, metrics.team_return_mean, alpha=0.5,
                    label="sampled (with exploration)")
            trained = evaluation(key)
            trained = trained[trained.policy == "trained"]
            ax.plot(trained.env_steps, trained.team_return_mean, "o-",
                    label="evaluation (most likely action)")
        ax.axhline(baselines["random"], color="grey", ls="--", label="random")
        ax.axhline(baselines["zero"], color="tab:red", ls=":", label="zero action")
        ax.set_title(f"{algorithm.upper()} {paradigm.upper()}", fontsize=10)
        if row == len(PARADIGMS) - 1:
            ax.set_xlabel("environment steps")
        if col == 0:
            ax.set_ylabel("team return")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()

### Learner health

Policy entropy for PPO and APPO (should fall as the policies sharpen, never blow
up) and SAC's temperature α, each averaged over the run's modules (three in DTDE,
one otherwise).

In [ ]:
fig, axes = plt.subplots(1, len(ALGORITHMS), squeeze=False, figsize=(5 * len(ALGORITHMS), 3.5))
axes = axes[0]
for ax, algorithm in zip(axes, ALGORITHMS):
    column = "/alpha_value" if algorithm == "sac" else "/entropy"
    for paradigm in PARADIGMS:
        key = (algorithm, paradigm)
        if key not in done:
            continue
        metrics = pd.read_csv(Path(RUNS[key].out_dir) / "metrics.csv")
        values = metrics.filter(like=column).mean(axis=1)
        ax.plot(metrics.env_steps, values, label=paradigm.upper())
    ax.set_title(f"{algorithm.upper()}: {'temperature alpha' if algorithm == 'sac' else 'policy entropy'}")
    ax.set_xlabel("environment steps")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

## d. Watching the agents

The same episode played by random actions and by trained policies (most likely
action). Agents that learned to spread each move to a different landmark and stay
there. First DTDE-PPO, then the combination with the best evaluation return.

In [ ]:
def frames_of(key):
    policy, _ = load_policy(RUNS[key].out_dir)
    frames, total = episode_frames(ENV_CONFIG, policy, SEED)
    return frames, total


if ("ppo", "dtde") in done:
    frames, total = frames_of(("ppo", "dtde"))
    print(f"DTDE PPO: team return {total:.1f} on this episode, random {return_random:.1f}")
    display(gif([frames_random, frames], ["random", "DTDE PPO (trained)"]))

best = max(done, key=lambda k: table.loc[k, "trained"])
print(f"best evaluation return: {best[0].upper()} {best[1].upper()}, {table.loc[best, 'trained']:.1f}")
if best != ("ppo", "dtde"):
    frames, total = frames_of(best)
    display(gif([frames_random, frames], ["random", f"{best[1].upper()} {best[0].upper()} (trained)"]))

## e. How to read this

*(Filled in once the full run is done.)*